# TF-IDF segmentado + word clouds por segmento

Este notebook responde una pregunta puntual: **¿qué vocabulario es característico de cada grupo de tickets, y no solo frecuente en general?**

Compara el enfoque de frecuencia simple (que ya sabemos que tiene un techo bajo: lo más frecuente no es necesariamente lo más importante) con TF-IDF, que identifica términos que distinguen a un grupo del resto del corpus.

## Nota de confidencialidad

Este notebook **corre sobre el texto original de los tickets (asunto + descripción), que NO está en este repositorio** por motivos de confidencialidad — igual que `notebook/nlp/topic_sentiment_analysis.ipynb`. Corré este notebook localmente con el archivo de texto real. Lo único que se publica al repo son **resultados agregados** (términos, scores, `n` por segmento) — nunca el texto original ni ejemplos de tickets individuales.


## Columnas esperadas

Este notebook asume que tenés un DataFrame con (al menos) estas columnas — **ajustá los nombres a los tuyos reales**, están marcados con `# AJUSTAR` donde corresponde:

| Columna esperada | Qué contiene |
|---|---|
| `texto` | asunto + descripción del ticket, texto libre |
| `tipo` | `Correctivo` / `Evolutivo` |
| `sentimiento` | `NEG` / `POS` / `NEU` (ya calculado en `topic_sentiment_analysis.ipynb`) |
| `modulo` | módulo/categoría del ticket |


In [ ]:
import pandas as pd
import re
import unicodedata
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 120)


## 1. Carga de datos

Cargá acá tu archivo local con el texto original. **Este archivo NO se commitea al repo.**
Si ya tenés el sentimiento calculado en otro notebook (`topic_sentiment_analysis.ipynb`),
hacé el merge por el id del ticket antes de este paso.


In [ ]:
# Columnas reales confirmadas en el export de Redmine (validado con el archivo real):
# 'Asunto', 'Descripción', 'Categoría' (Correctivo/Evolutivo/Migración/Jasper),
# 'Proyecto' (funciona como "módulo": Gestión Economica, Servicios del Centro, Ventanilla, etc.)
# Ojo: el separador es ";" (no ","), y trae comillas dobles envolviendo casi todos los campos.

RUTA_DATOS_PRIVADOS = "../../data_privado/tickets_redmine_original.csv"  # AJUSTAR ruta local

df = pd.read_csv(
    RUTA_DATOS_PRIVADOS,
    sep=';',
    quotechar='"',
    encoding='utf-8-sig',
    engine='python',
)

df['texto'] = df['Asunto'].fillna('') + ' ' + df['Descripción'].fillna('')

print(f"Total de tickets cargados: {len(df)}")
print(f"Tickets con descripcion no vacia: {df['Descripción'].notna().sum()}")
df[['Asunto', 'Categoría', 'Proyecto']].head(3)


## 2. Preprocessing

Limpieza estándar: minúsculas, sin tildes (para no duplicar vocabulario por acentuación
inconsistente), sin puntuación ni números, sin stopwords en español.


In [ ]:
# Nombres propios a excluir (Autor / Asignado a) -- proteccion extra de privacidad,
# ademas del anonimizado que ya recibe el dataset original antes de llegar aca.
nombres_a_excluir = set()
for col in ['Autor', 'Asignado a']:
    for val in df[col].dropna().unique():
        val = str(val)
        if val.strip() in ('""', ''):
            continue
        for tok in val.lower().split():
            tok = re.sub(r'[^a-zA-Z]', '', tok)
            tok = ''.join(c for c in unicodedata.normalize('NFD', tok) if unicodedata.category(c) != 'Mn')
            if len(tok) > 2:
                nombres_a_excluir.add(tok.lower())

print(f"Terminos de nombres propios a excluir: {len(nombres_a_excluir)}")

# Variantes catalan/castellano detectadas en una primera corrida (el dataset mezcla ambos idiomas)
VARIANTES_CA_ES = {
    'gestio': 'gestion', 'informacio': 'informacion', 'peticio': 'peticion',
    'validacio': 'validacion', 'autoritzacio': 'autorizacion', 'centre': 'centro',
}

STOPWORDS_ES_BASE = {
    "de","la","que","el","en","y","a","los","del","se","las","por","un","para",
    "con","no","una","su","al","lo","como","mas","pero","sus","le","ya","o",
    "este","si","porque","esta","entre","cuando","muy","sin","sobre","tambien",
    "me","hasta","hay","donde","quien","desde","todo","nos","durante","todos",
    "uno","les","ni","contra","otros","ese","eso","ante","ellos","e","esto",
    "mi","antes","algunos","unos","yo","otro","otras","otra","tanto",
    "esa","estos","mucho","quienes","nada","muchos","cual","poco","ella","estar",
    "estas","algunas","algo","nosotros","mio","tuyo","ellas","nosotras","vosotros",
    "vosotras","os","tuya","tuyos","tuyas","nuestro","nuestra","nuestros",
    "nuestras","vuestro","vuestra","vuestros","vuestras","esos","esas",
    "buenos","dias","tardes","favor","gracias","saludos","cordiales","atentamente",
    "hola","perdon","perdona","disculpen","disculpa","adjunto","adjuntamos",
    "gestion","llull","sigp","aa","dgpc","caib",
    # ruido de formato / codigos detectado en la primera corrida sobre datos reales:
    "png","jpg","jpeg","pdf","doc","docx","xlsx","xls","req","serv","com",
}

def normalizar(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    palabras = []
    for w in texto.split():
        w = VARIANTES_CA_ES.get(w, w)
        if w in STOPWORDS_ES_BASE or w in nombres_a_excluir or len(w) <= 2:
            continue
        palabras.append(w)
    return ' '.join(palabras)

df['texto_limpio'] = df['texto'].apply(normalizar)
df[['texto_limpio']].head(3)


## 3. Definición de segmentos

Cada segmento se define como una máscara booleana. Se reporta siempre el `n`
(cantidad de tickets del segmento) junto a los resultados — es la corrección
más importante frente a la versión simple del análisis: con 310 tickets totales,
algunos cruces van a tener muestras chicas, y eso tiene que ser visible, no
implícito.


In [ ]:
# Columnas y valores reales confirmados sobre el archivo original.
# La columna "sentimiento" todavia no esta acá: se calcula en topic_sentiment_analysis.ipynb
# por separado. Si ya la corriste y tenés un archivo local con sentimiento por ticket,
# mergealo a "df" por el id ("#") antes de esta celda y descomentá esas dos filas.

segmentos = {
    "Correctivo":          df['Categoría'] == 'Correctivo',
    "Evolutivo":           df['Categoría'] == 'Evolutivo',
    "Gestion economica":   df['Proyecto'] == 'Gestión Economica',
    "Resto de modulos":    df['Proyecto'] != 'Gestión Economica',
    # "Sentimiento negativo":        df['sentimiento'] == 'NEG',
    # "Sentimiento positivo/neutro": df['sentimiento'].isin(['POS', 'NEU']),
}

UMBRAL_N_MINIMO = 15

resumen_n = pd.DataFrame({
    "segmento": list(segmentos.keys()),
    "n": [mask.sum() for mask in segmentos.values()],
})
resumen_n["confiable"] = resumen_n["n"] >= UMBRAL_N_MINIMO
resumen_n


## 4. TF-IDF global y términos característicos por segmento

Se ajusta un único `TfidfVectorizer` sobre todo el corpus (para que el vocabulario
y el IDF sean comparables entre segmentos), y después se calcula, para cada
segmento, el **promedio de TF-IDF por término entre los documentos de ese
segmento**. Los términos con mayor promedio son los característicos del grupo
— no simplemente los más frecuentes.


In [ ]:
vectorizer = TfidfVectorizer(
    max_df=0.85,     # descarta terminos presentes en mas del 85% de los tickets (demasiado genericos)
    min_df=2,        # descarta terminos que aparecen en un unico ticket (ruido / typos)
    ngram_range=(1, 2),  # incluye unigramas y bigramas ("factura pendiente", no solo "factura")
)

tfidf_matrix = vectorizer.fit_transform(df['texto_limpio'])
vocabulario = vectorizer.get_feature_names_out()

print(f"Vocabulario: {len(vocabulario)} terminos/bigramas")


In [ ]:
def terminos_caracteristicos(mask, top_n=12):
    n = mask.sum()
    if n == 0:
        return pd.DataFrame(columns=['termino', 'tfidf_promedio']), 0

    submatriz = tfidf_matrix[mask.values]
    promedio_por_termino = submatriz.mean(axis=0).A1  # promedio de cada columna (termino)

    top_idx = promedio_por_termino.argsort()[::-1][:top_n]
    resultado = pd.DataFrame({
        'termino': vocabulario[top_idx],
        'tfidf_promedio': promedio_por_termino[top_idx],
    })
    return resultado, n

resultados_por_segmento = {}
for nombre_segmento, mask in segmentos.items():
    tabla, n = terminos_caracteristicos(mask)
    resultados_por_segmento[nombre_segmento] = {'tabla': tabla, 'n': n}
    print(f"\n=== {nombre_segmento} (n={n}) ===")
    if n < UMBRAL_N_MINIMO:
        print(f"  ATENCION: n={n} esta por debajo del umbral de {UMBRAL_N_MINIMO}. Resultado poco confiable.")
    print(tabla.to_string(index=False))


## 5. Tabla para el README

Genera el markdown listo para pegar en la sección "Pipeline NLP" del README
(reemplaza los placeholders `*(completar tras correr el análisis)*`).


In [ ]:
def top_terminos_str(tabla, k=6):
    return ', '.join(tabla['termino'].head(k).tolist())

filas_md = []
filas_md.append("| Segmento | n | Términos característicos |")
filas_md.append("|---|---|---|")
for nombre_segmento, datos in resultados_por_segmento.items():
    n = datos['n']
    marca_n = f"{n}" if n >= UMBRAL_N_MINIMO else f"{n} ⚠️"
    filas_md.append(f"| {nombre_segmento} | {marca_n} | {top_terminos_str(datos['tabla'])} |")

tabla_markdown = '\n'.join(filas_md)
print(tabla_markdown)


In [ ]:
# Exporta SOLO datos agregados (terminos + scores + n) -- nunca texto original ni tickets individuales
filas_export = []
for nombre_segmento, datos in resultados_por_segmento.items():
    tabla = datos['tabla'].copy()
    tabla['segmento'] = nombre_segmento
    tabla['n_segmento'] = datos['n']
    filas_export.append(tabla)

export_df = pd.concat(filas_export, ignore_index=True)[['segmento', 'n_segmento', 'termino', 'tfidf_promedio']]

# AJUSTAR ruta: esta si se publica al repo (no contiene texto original, solo terminos agregados)
export_df.to_csv('../../data/tfidf_terminos_por_segmento.csv', index=False)
export_df.head(10)


## 6. Word clouds por segmento

Solo para segmentos con `n >= UMBRAL_N_MINIMO`. Las nubes se generan a partir
de los **pesos TF-IDF** (`generate_from_frequencies`), no de la frecuencia
simple — así la nube es una visualización de la misma tabla de arriba, no una
pieza decorativa aparte.


In [ ]:
segmentos_a_graficar = {
    nombre: datos for nombre, datos in resultados_por_segmento.items()
    if datos['n'] >= UMBRAL_N_MINIMO
}

n_graficos = len(segmentos_a_graficar)
cols = 2
filas = (n_graficos + 1) // cols

fig, axes = plt.subplots(filas, cols, figsize=(14, 5 * filas))
axes = axes.flatten() if n_graficos > 1 else [axes]

for ax, (nombre_segmento, datos) in zip(axes, segmentos_a_graficar.items()):
    frecuencias = dict(zip(datos['tabla']['termino'], datos['tabla']['tfidf_promedio']))
    wc = WordCloud(
        width=800, height=500,
        background_color='white',
        colormap='viridis',
        max_words=30,
    ).generate_from_frequencies(frecuencias)

    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f"{nombre_segmento} (n={datos['n']})", fontsize=13, fontweight='bold')
    ax.axis('off')

# oculta ejes sobrantes si el numero de segmentos es impar
for ax in axes[n_graficos:]:
    ax.axis('off')

plt.tight_layout()
plt.savefig('../../assets/wordclouds_por_segmento.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nRevisa la imagen antes de publicarla: si algun termino te parece")
print("demasiado especifico (nombre propio, numero de expediente, etc.),")
print("agregalo a STOPWORDS_ES_BASE en la celda de preprocessing y volve a correr.")


## Próximo paso

1. Corré este notebook localmente con tu archivo de texto real.
2. Copiá la tabla markdown de la sección 5 al README de `redmine-ops-analytics`,
   reemplazando los placeholders.
3. Subí `assets/wordclouds_por_segmento.png` y `data/tfidf_terminos_por_segmento.csv`
   al repo (ambos son agregados, no contienen texto original).
4. Marcá el ítem correspondiente como hecho en "Próximos pasos" del README.
